In [26]:
import MySQLdb

In [27]:
class Word:
    def __init__(self, eng, kor, lev=1):
        self.eng = eng
        self.kor = kor
        self.lev = lev

    def __repr__(self):
        return f"Word(eng='{self.eng}', kor='{self.kor}', lev={self.lev})"

    @property
    def eng(self):
        return self.__eng

    @eng.setter
    def eng(self, eng):
        if not eng:
            raise ValueError("영어 단어는 비워둘 수 없습니다")
        self.__eng = eng

    @property
    def kor(self):
        return self.__kor

    @kor.setter
    def kor(self, kor):
        if not kor:
            raise ValueError("뜻을 비워둘 수 없습니다")
        self.__kor = kor

    @property
    def lev(self):
        return self.__lev

    @lev.setter
    def lev(self, lev):
        lev = int(lev)
        if lev < 1:
            raise ValueError("레벨은 1 이상이어야 합니다")
        self.__lev = lev

In [28]:
class WordsDAO: # Data Access Object
    def __init__(self):
        self.db = None

    def connect(self):
        self.db = MySQLdb.connect(host='localhost', user='apple', password='1111', db='ai', charset='utf8')

    def disconnect(self):
        if self.db:
            self.db.close()

    def insert(self, word):
        self.connect()
        cur = self.db.cursor()
        sql = "insert into voca (eng, kor, lev) values (%s, %s, %s)"
        data = (word.eng, word.kor, word.lev)
        cur.execute(sql, data)
        self.db.commit()
        cur.close()
        self.disconnect()

    def select_all(self):
        self.connect()
        cur = self.db.cursor(MySQLdb.cursors.DictCursor)
        sql = "SELECT eng, kor, lev FROM voca ORDER BY eng ASC"
        cur.execute(sql)
        rows = cur.fetchall()
        cur.close()
        self.disconnect()
        return rows


    def search_all(self, eng):
        self.connect()
        cur = self.db.cursor(MySQLdb.cursors.DictCursor)
        sql = "SELECT eng, kor, lev FROM voca WHERE eng LIKE CONCAT('%%', %s, '%%')"
        cur.execute(sql, (eng,))
        rows = cur.fetchall()
        cur.close()
        self.disconnect()
        return rows
    
    def search(self, eng):
        self.connect()
        cur = self.db.cursor(MySQLdb.cursors.DictCursor)
        sql = "SELECT eng, kor, lev FROM voca WHERE eng = %s"
        cur.execute(sql, (eng,))
        row = cur.fetchone()
        cur.close()
        self.disconnect()
        return row
    

    def update(self, word):
        self.connect()
        cur = self.db.cursor()
        sql = "UPDATE voca SET kor=%s, lev=%s WHERE eng=%s"
        data = (word.kor, word.lev, word.eng)
        result = cur.execute(sql, data)
        self.db.commit()
        cur.close()
        self.disconnect()
        return result
    
    def delete(self, eng):
        self.connect()
        cur = self.db.cursor()
        sql = "DELETE FROM voca WHERE eng = %s"
        result = cur.execute(sql, (eng,))
        self.db.commit()
        cur.close()
        self.disconnect()
        return result


In [29]:
class WordsService:
    def __init__(self):
        self.dao = WordsDAO()

    def insert_word(self):
        eng = input("단어를 입력하세요: ")
        kor = input("뜻을 입력하세요: ")
        lev = input("레벨을 입력하세요: ")

        word = Word(eng, kor, lev)
        self.dao.insert(word)
        print("단어가 등록되었습니다")

    def print_all(self):
        words = self.dao.select_all()

        if not words:
            print("등록된 단어가 없다")
            return
        
        for word in words:
            print(f"{word['eng']} 뜻: {word['kor']}, 레벨: {word['lev']}")
    

    def search_words(self):
        eng = input("검색할 단어 입력: )")
        words = self.dao.search_all(eng)
        
        if not words:
            print("찾는 단어 없음")
            return
        
        for word in words:
            print(f"{word['eng']} 뜻: {word['kor']}, 레벨: {word['lev']}")


    def edit_word(self):
        eng = input("수정할 단어 입력")
        word = self.dao.search(eng)
        
        if not word:
            print("수정할 단어 없음")
            return
        
        kor = input("새로운 뜻 입력: ")
        lev = input("새로운 레벨 입력: ")
        word = Word(eng, kor, lev)

        result = self.dao.update(word)

        if result > 0:
            print('수정 성공')
        else:
            print('수정 실패')

    
    def delete_word(self):
        eng = input("삭제할 단어 입력: ")
        word = self.dao.search(eng)
        if not word:
            print("삭제할 단어 없음")
            return
        result = self.dao.delete(eng)

        if result > 0:
            print('삭제 성공')
        else:
            print('삭제 실패')
        

In [30]:
class Menu:
    def __init__(self):
        self.service = WordsService()

    def run(self):
        while True:
            try:
                print()
                print("===== 단어장 프로그램 =====")
                print("1. 등록하기")
                print("2. 출력하기")
                print("3. 검색하기")
                print("4. 수정하기")
                print("5. 삭제하기")
                print("6. 종료하기")
                menu = int(input("메뉴를 선택하세요"))

                if menu == 1:
                    print("단어를 등록합니다")
                    self.service.insert_word()

                elif menu == 2:
                    print("단어를 출력합니다")
                    self.service.print_all()
                    
                elif menu == 3:
                    print("단어를 검색합니다")
                    self.service.search_words()
                    
                elif menu == 4:
                    print("단어를 수정합니다")
                    self.service.edit_word()
                    
                elif menu == 5:
                    print("단어를 삭제합니다")
                    self.service.delete_word()
                    
                elif menu == 6:
                    print("프로그램을 종료합니다")
                    break
                else:
                    print("메뉴는 1부터 6까지 입력해주세요")
            except Exception as e:
                print("오류: ", e)
                print("다시 입력하세요")

In [32]:
menu = Menu()
menu.run()



===== 단어장 프로그램 =====
1. 등록하기
2. 출력하기
3. 검색하기
4. 수정하기
5. 삭제하기
6. 종료하기
단어를 삭제합니다
삭제할 단어 없음

===== 단어장 프로그램 =====
1. 등록하기
2. 출력하기
3. 검색하기
4. 수정하기
5. 삭제하기
6. 종료하기
단어를 삭제합니다
삭제 성공

===== 단어장 프로그램 =====
1. 등록하기
2. 출력하기
3. 검색하기
4. 수정하기
5. 삭제하기
6. 종료하기
프로그램을 종료합니다
